In [1]:
import os
import numpy as np
from datetime import datetime

save_dir = "./models/"
os.makedirs(save_dir, exist_ok=True)

In [2]:
# Register Atari environments
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

# Verify registration
atari_envs = [env_id for env_id in gym.envs.registry.keys() if env_id.startswith("ALE/")]
print("Number of ALE envs:", len(atari_envs))
print("Sample envs:", atari_envs[:10])

# Smoke test
env = gym.make("ALE/Breakout-v5")
obs, info = env.reset()
print("Observation shape:", obs.shape)
env.close()

Number of ALE envs: 104
Sample envs: ['ALE/Adventure-v5', 'ALE/AirRaid-v5', 'ALE/Alien-v5', 'ALE/Amidar-v5', 'ALE/Assault-v5', 'ALE/Asterix-v5', 'ALE/Asteroids-v5', 'ALE/Atlantis2-v5', 'ALE/Atlantis-v5', 'ALE/Backgammon-v5']
Observation shape: (210, 160, 3)


A.L.E: Arcade Learning Environment (version 0.11.2+unknown)
[Powered by Stella]


In [3]:
gym.register_envs(ale_py)

from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3 import DQN

vec_env = make_atari_env(
    "ALE/Galaxian-v5",
    n_envs=4,
    seed=0,
    wrapper_kwargs=dict(terminal_on_life_loss=False)
)

vec_env = VecFrameStack(vec_env, n_stack=4)

In [4]:
# need to shrink replay buffer to save RAM
# also, control exploration manually in learning loop so that we can break up training
model = DQN("CnnPolicy", vec_env, verbose=1, buffer_size=400_000, exploration_initial_eps=1.00, exploration_fraction=50.0)

Using cpu device
Wrapping the env in a VecTransposeImage.


In [5]:
chunk_size = 20_000
num_chunks = 50

for i in range(1, num_chunks + 1):
    model.learn(
        total_timesteps=chunk_size,
        reset_num_timesteps=(i == 1),
        log_interval=100
    )

    total_so_far = i * chunk_size

    model_path = os.path.join(save_dir, f"dqn_2_pong_model_{total_so_far}")
    model.save(model_path)
    print("Model saved to:", model_path + ".zip")

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 622      |
|    ep_rew_mean      | 677      |
|    exploration_rate | 0.986    |
| time/               |          |
|    episodes         | 100      |
|    fps              | 317      |
|    time_elapsed     | 47       |
|    total_timesteps  | 15208    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0224   |
|    n_updates        | 944      |
----------------------------------
Model saved to: ./models/dqn_2_pong_model_20000.zip
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 595      |
|    ep_rew_mean      | 613      |
|    exploration_rate | 0.986    |
| time/               |          |
|    episodes         | 200      |
|    fps              | 327      |
|    time_elapsed     | 29       |
|    total_timesteps  | 29524    |
| train/              |          |
|    learning_rate    | 0.0001   |
|  

In [6]:
# record video of agent

video_folder = "videos/"

os.makedirs(video_folder, exist_ok=True)
fps = vec_env.metadata.get("render_fps", 30)
video_name = model_path.split('/')[-1]

recorded_frames = []

import moviepy

In [7]:
# record 100 frames
for i in range(1000):
    vec_env.step_wait()
    frame = vec_env.render()
    if (isinstance(frame, np.ndarray)):
        recorded_frames.append(frame)
    else:
        print("uh oh!")
        show(frame)

In [8]:
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip

clip = ImageSequenceClip(recorded_frames, fps=fps)
clip.write_videofile(os.path.join(video_folder, video_name) + ".mp4")

# del clip
# del recorded_frames

MoviePy - Building video videos/dqn_2_pong_model_1000000.mp4.
MoviePy - Writing video videos/dqn_2_pong_model_1000000.mp4



MoviePy - Done !
MoviePy - video ready videos/dqn_2_pong_model_1000000.mp4


In [ ]:
model.gamma = 0.9

## Notes on Continuation

1. initial: 2 million iterations with default settings, except for shrunk replay buffer (necessary to make the algorithm fit in my machine's RAM)
2. 2m-4m: continue with same settings
3. 4m-6.2m: continue with same settings (considered increasing exploration values, but didn't)
4. 6.2m-8.2m: continue with gamma decreased to 0.99 -> 0.9


In [ ]:
model = DQN.load(os.path.join(model_path, "dqn_pong_model_6200000.zip"), env=vec_env)

In [ ]:
chunk_size = 20_000
num_chunks = 100

for i in range(1, num_chunks + 1):
    model.learn(
        total_timesteps=chunk_size,
        reset_num_timesteps=(i == 1),
        log_interval=100
    )

    total_so_far = i * chunk_size + 6_200_000

    model_path = os.path.join(save_dir, f"dqn_pong_model_{total_so_far}")
    model.save(model_path)
    print("Model saved to:", model_path + ".zip")

# A2C

In [1]:
import os
import numpy as np
from datetime import datetime

save_dir = "./models/"
os.makedirs(save_dir, exist_ok=True)

In [2]:
# Register Atari environments
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

# Verify registration
atari_envs = [env_id for env_id in gym.envs.registry.keys() if env_id.startswith("ALE/")]
print("Number of ALE envs:", len(atari_envs))
print("Sample envs:", atari_envs[:10])

# Smoke test
env = gym.make("ALE/Breakout-v5")
obs, info = env.reset()
print("Observation shape:", obs.shape)
env.close()

Number of ALE envs: 104
Sample envs: ['ALE/Adventure-v5', 'ALE/AirRaid-v5', 'ALE/Alien-v5', 'ALE/Amidar-v5', 'ALE/Assault-v5', 'ALE/Asterix-v5', 'ALE/Asteroids-v5', 'ALE/Atlantis2-v5', 'ALE/Atlantis-v5', 'ALE/Backgammon-v5']
Observation shape: (210, 160, 3)


A.L.E: Arcade Learning Environment (version 0.11.2+unknown)
[Powered by Stella]


In [3]:
gym.register_envs(ale_py)

from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3 import A2C

vec_env = make_atari_env(
    "ALE/Galaxian-v5",
    n_envs=4,
    seed=0,
    wrapper_kwargs=dict(terminal_on_life_loss=False),
)

# model gets 4 frames (?)
vec_env = VecFrameStack(vec_env, n_stack=4)

In [ ]:
model = A2C("CnnPolicy", vec_env, verbose=1, explore)

In [ ]:
chunk_size = 80_000
num_chunks = 100

for i in range(1, num_chunks + 1):
    model.learn(
        total_timesteps=chunk_size,
        reset_num_timesteps=(i == 1),
        log_interval=100
    )

    total_so_far = i * chunk_size

    model_path = os.path.join(save_dir, f"a2c_2_pong_model_{total_so_far}")
    model.save(model_path)
    print("Model saved to:", model_path + ".zip")

In [ ]:
# record video of agent

video_folder = "videos/"

os.makedirs(video_folder, exist_ok=True)
fps = vec_env.metadata.get("render_fps", 30)
video_name = model_path.split('/')[-1]

recorded_frames = []

import moviepy

In [ ]:
# record 100 frames
for i in range(1000):
    vec_env.step_wait()
    frame = vec_env.render()
    if (isinstance(frame, np.ndarray)):
        recorded_frames.append(frame)
    else:
        print("uh oh!")
        show(frame)

In [ ]:
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip

clip = ImageSequenceClip(recorded_frames, fps=fps)
clip.write_videofile(os.path.join(video_folder, video_name) + ".mp4")

# del clip
# del recorded_frames

# PPO

In [1]:
import os
import numpy as np
from datetime import datetime

save_dir = "./models/"
os.makedirs(save_dir, exist_ok=True)

In [2]:
# Register Atari environments
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

# Verify registration
atari_envs = [env_id for env_id in gym.envs.registry.keys() if env_id.startswith("ALE/")]
print("Number of ALE envs:", len(atari_envs))
print("Sample envs:", atari_envs[:10])

# Smoke test
env = gym.make("ALE/Breakout-v5")
obs, info = env.reset()
print("Observation shape:", obs.shape)
env.close()

Number of ALE envs: 104
Sample envs: ['ALE/Adventure-v5', 'ALE/AirRaid-v5', 'ALE/Alien-v5', 'ALE/Amidar-v5', 'ALE/Assault-v5', 'ALE/Asterix-v5', 'ALE/Asteroids-v5', 'ALE/Atlantis2-v5', 'ALE/Atlantis-v5', 'ALE/Backgammon-v5']
Observation shape: (210, 160, 3)


A.L.E: Arcade Learning Environment (version 0.11.2+unknown)
[Powered by Stella]


In [3]:
gym.register_envs(ale_py)

from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

vec_env = make_atari_env(
    "ALE/Galaxian-v5",
    n_envs=4,
    seed=0,
    wrapper_kwargs=dict(terminal_on_life_loss=False)
)

# model gets 4 frames (?)
vec_env = VecFrameStack(vec_env, n_stack=4)

In [4]:
# need to shrink replay buffer to save RAM
from stable_baselines3 import PPO
model = PPO("CnnPolicy", vec_env, verbose=1)
                   

Using cpu device
Wrapping the env in a VecTransposeImage.


In [5]:
chunk_size = 20_000
num_chunks = 400

for i in range(1, num_chunks + 1):
    model.learn(
        total_timesteps=chunk_size,
        reset_num_timesteps=(i == 1),
        log_interval=100
    )

    total_so_far = i * chunk_size

    model_path = os.path.join(save_dir, f"ppo_pong_model_{total_so_far}")
    model.save(model_path)
    print("Model saved to:", model_path + ".zip")

Model saved to: ./models/ppo_pong_model_20000.zip
Model saved to: ./models/ppo_pong_model_40000.zip
Model saved to: ./models/ppo_pong_model_60000.zip
Model saved to: ./models/ppo_pong_model_80000.zip
Model saved to: ./models/ppo_pong_model_100000.zip
Model saved to: ./models/ppo_pong_model_120000.zip
Model saved to: ./models/ppo_pong_model_140000.zip
Model saved to: ./models/ppo_pong_model_160000.zip
Model saved to: ./models/ppo_pong_model_180000.zip
Model saved to: ./models/ppo_pong_model_200000.zip
Model saved to: ./models/ppo_pong_model_220000.zip
Model saved to: ./models/ppo_pong_model_240000.zip
Model saved to: ./models/ppo_pong_model_260000.zip
Model saved to: ./models/ppo_pong_model_280000.zip
Model saved to: ./models/ppo_pong_model_300000.zip
Model saved to: ./models/ppo_pong_model_320000.zip
Model saved to: ./models/ppo_pong_model_340000.zip
Model saved to: ./models/ppo_pong_model_360000.zip
Model saved to: ./models/ppo_pong_model_380000.zip
Model saved to: ./models/ppo_pong_m

In [6]:
# record video of agent

video_folder = "videos/"

os.makedirs(video_folder, exist_ok=True)
fps = vec_env.metadata.get("render_fps", 30)
video_name = model_path.split('/')[-1]

recorded_frames = []

import moviepy

In [7]:
# record 100 frames
for i in range(1000):
    vec_env.step_wait()
    frame = vec_env.render()
    if (isinstance(frame, np.ndarray)):
        recorded_frames.append(frame)
    else:
        print("uh oh!")
        show(frame)

In [8]:
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip

clip = ImageSequenceClip(recorded_frames, fps=fps)
clip.write_videofile(os.path.join(video_folder, video_name) + ".mp4")

# del clip
# del recorded_frames

MoviePy - Building video videos/ppo_pong_model_8000000.mp4.
MoviePy - Writing video videos/ppo_pong_model_8000000.mp4



MoviePy - Done !
MoviePy - video ready videos/ppo_pong_model_8000000.mp4
